# Vẽ biểu đồ Learning Curve cho mô hình Random Forest (Pipeline Mới)

Quy trình:
1. Load dữ liệu đã xử lý (`X.parquet`, `y.parquet`).
2. Lấy ML Pipeline từ Backend (tự động Impute & Encode).
3. Sử dụng `learning_curve` từ sklearn để tính toán độ chính xác.
4. Trực quan hóa kết quả.

In [ ]:
import pandas as pd
import numpy as np
# pyrefly: ignore [missing-import]
import matplotlib.pyplot as plt
from sklearn.model_selection import learning_curve
import os
import sys
from pathlib import Path

# Thêm đường dẫn backend vào sys.path để import
current_file = Path(__file__).resolve()
project_root = current_file.parents[1]
sys.path.append(str(project_root / "backend"))

from app.ml.pipeline import build_pipeline, load_best_params

# Cấu hình đường dẫn
PROCESSED_DIR = project_root / 'data' / 'processed'
REPORT_DIR = project_root / 'reports' / 'figures'
os.makedirs(REPORT_DIR, exist_ok=True)

In [ ]:
# 1. Load Data
print("Loading data...")
X = pd.read_parquet(PROCESSED_DIR / 'X.parquet')
y = pd.read_parquet(PROCESSED_DIR / 'y.parquet')['target']

In [ ]:
# 2. Xây dựng Pipeline
print("Building pipeline...")
best_params = load_best_params()
# Loại bỏ random_state hoặc n_jobs từ best_params nếu có để tránh trùng lặp khi truyền vào learning_curve
rf_kwargs = {k: v for k, v in best_params.items() if k not in ["random_state", "n_jobs"]}
pipeline = build_pipeline(rf_kwargs)

In [ ]:
# 3. Tính toán Learning Curve
print("Calculating learning curve... This might take a moment.")
# learning_curve sẽ fit toàn bộ pipeline (bao gồm cả bước chuẩn hóa One-Hot/Ordinal)
train_sizes, train_scores, test_scores = learning_curve(
    pipeline,
    X, y, 
    cv=5, 
    n_jobs=-1, 
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='accuracy',
    error_score='raise'
)

# Tính trung bình và độ lệch chuẩn
train_scores_mean = np.mean(train_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
test_scores_mean = np.mean(test_scores, axis=1)
test_scores_std = np.std(test_scores, axis=1)

In [ ]:
# 4. Trực quan hóa 
plt.figure(figsize=(10, 6))
plt.title("Learning Curve (Random Forest + ML Pipeline)")
plt.xlabel("Sample Size (Training examples)")
plt.ylabel("Accuracy")
plt.grid()

# Vẽ dải độ lệch chuẩn
plt.fill_between(train_sizes, train_scores_mean - train_scores_std,
                 train_scores_mean + train_scores_std, alpha=0.1, color="r")
plt.fill_between(train_sizes, test_scores_mean - test_scores_std,
                 test_scores_mean + test_scores_std, alpha=0.1, color="g")

# Vẽ đường trung bình
plt.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Training score")
plt.plot(train_sizes, test_scores_mean, 'o-', color="g", label="Cross-validation score")

plt.legend(loc="best")
plt.tight_layout()

# Lưu biểu đồ
output_path = REPORT_DIR / 'learning_curve.png'
plt.savefig(output_path)
print(f"Learning curve saved to: {output_path}")
# plt.show()